In [1]:
from utils import *

We avoid permutational equivalent classes by testing different confounding structures

In [ ]:
conf_str = [
    [(1,2)],                              # 1 edge
    [(1,2),(1,3)],                        # 2 edges
    [(1,2),(1,3),(1,4)],                  # 3 edges: star
    [(1,2),(1,3),(2,3)],                  # 3 edges: triangle + isolated
    [(1,2),(1,3),(2,4)],                  # 3 edges: path P4
    [(1,2),(1,3),(2,3),(1,4)],            # 4 edges: triangle + pendant
    [(1,2),(2,3),(3,4),(4,1),(1,3)],      # 5 edges: 4-cycle with diagonal
    [(1,2),(1,3),(1,4),(2,3),(2,4)],      # 5 edges: K4 minus one edge
]

In [4]:
def all_dags(n):
    vertices = range(1, n + 1)

    edges = [(i, j) for i in vertices for j in vertices if i != j]

    dags = []

    for mask in range(1 << len(edges)):
        G = nx.DiGraph()
        G.add_nodes_from(vertices)

        for k, edge in enumerate(edges):
            if mask & (1 << k):
                G.add_edge(*edge)

        if nx.is_directed_acyclic_graph(G):
            dags.append(set(G.edges()))

    return dags

dags = all_dags(4)

Find cases where the equivalence class by exhaustive search differs from the one by DFS/BFS (adding/removing edges)

In [ ]:
n = 4
confounding_count = 1
stop = False
num_bad_cases = 0
tested_cases = 0
all_bad_cases = []

for B in conf_str:

    dags_copy = dags

    while len(dags_copy) > 0:
        G_temp = dags_copy[0]
        # The general idea is to run exhaustive search vs run DFS/BFS
        res_trav = find_equivalence_class_traversal(G_temp,B,n)
        res_exh = find_equivalent_graphs_exhaustive_to_compare(G_temp,B,n)

        if frozenset(map(frozenset, res_trav)) != frozenset(map(frozenset, res_exh)):
            all_bad_cases.append(res_trav)
            num_bad_cases = num_bad_cases + 1

        dags_copy = [x for x in dags_copy if x not in res_exh]
        tested_cases = tested_cases + 1
        print(f"Remaining DAG: {len(dags_copy)}/{len(dags)}, Confounding id: {confounding_count}/{len(conf_str)}, Tested Cases: {tested_cases}, Bad Cases: {num_bad_cases}")

    confounding_count = confounding_count + 1

Display bad cases:

In [ ]:
all_bad_cases